# Train Attention-Based MIL with Regularization (Colab)

Extends the baseline MIL notebook with three lightweight regularization options:
- **Attention entropy regularization** — penalizes concentrated attention (addresses the unregularized attention mechanism)
- **Gradient clipping** — stabilizes training across variable bag sizes (652–37K patches)
- **Label smoothing** — reduces overconfidence for downstream confidence-based routing

All regularization defaults to disabled (`0.0`), preserving baseline behavior.

Includes a **Compare Mode** cell that trains baseline and regularized on a single fold
with the same split, producing side-by-side diagnostic plots.

**Requirements:** GPU runtime (T4 or L4), embeddings on Drive at `/content/drive/MyDrive/prame-predict/embeddings/`

In [ ]:
# Setup: mount Drive, install deps, clone repo for manifest
from google.colab import drive
drive.mount("/content/drive")

!pip install -q h5py scikit-learn matplotlib pandas

import subprocess, shutil
if not __import__("pathlib").Path("/content/prame-predict").exists():
    subprocess.run(["git", "clone", "https://github.com/hb-1968/prame-predict.git"],
                   cwd="/content", check=True)
print("Setup complete.")

In [ ]:
import json
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve
from pathlib import Path
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

## Configuration

In [ ]:
# --- Paths ---
DRIVE_EMB_DIR = Path("/content/drive/MyDrive/prame-predict/embeddings")
MANIFEST_PATH = Path("/content/prame-predict/data/expression/slide_manifest.csv")
RESULTS_DIR = Path("/content/drive/MyDrive/prame-predict/results")

# --- Which model embeddings to train on ---
MODEL = "uni"  # "uni" (1024-dim) or "conch" (768-dim)

# --- Hyperparameters ---
FOLDS = 5
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
HIDDEN_DIM = 256
ATTN_DIM = 128
DROPOUT = 0.25
PATIENCE = 10  # early stopping patience
SEED = 42

# --- Regularization (set to 0.0 to disable) ---
# Post-ablation heuristic (fold 1, 100 slides):
#   ENTROPY_LAMBDA : inert at gentle coefficients -- leave at 0.0
#   GRAD_CLIP      : lowers overfitting gap, keep at 1.0
#   LABEL_SMOOTHING: only reg with positive AUC delta, keep at 0.05
ENTROPY_LAMBDA = 0.0   # kept for experimentation; ablation found it inert
GRAD_CLIP = 0.0        # recommended production value: 1.0
LABEL_SMOOTHING = 0.0  # recommended production value: 0.05

# --- Slide limit (0 = use all) ---
MAX_SLIDES = 0  # stratified subsample (try 100 for faster CPU runs)

# --- Derived ---
FEAT_DIM = {"uni": 1024, "conch": 768}[MODEL]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Model: {MODEL.upper()} | feat_dim={FEAT_DIM} | {FOLDS}-fold CV | patience={PATIENCE}")
if ENTROPY_LAMBDA > 0 or GRAD_CLIP > 0 or LABEL_SMOOTHING > 0:
    print(f"Regularization: entropy_lambda={ENTROPY_LAMBDA}, grad_clip={GRAD_CLIP}, label_smoothing={LABEL_SMOOTHING}")
else:
    print("Regularization: disabled (baseline mode)")
if MAX_SLIDES > 0:
    print(f"Slide limit: {MAX_SLIDES}")

## Dataset & Model

In [ ]:
class GPUCachedSlideDataset(Dataset):
    """Load all patch embeddings into GPU VRAM once (fp32). Zero I/O per epoch."""

    def __init__(self, slide_paths, labels, device):
        self.labels = [torch.tensor(y, dtype=torch.float32, device=device)
                       for y in labels]
        self.bags = []
        total = 0
        for p in slide_paths:
            with h5py.File(p, "r") as f:
                arr = f["features"][:]  # native fp16 on disk
            # Upcast to fp32 to match model weights (nn.Linear requires matching dtypes).
            t = torch.from_numpy(arr).to(device=device, dtype=torch.float32)
            self.bags.append(t)
            total += t.element_size() * t.nelement()
        print(f"  Cached {len(self.bags)} bags in VRAM "
              f"({total / 1e9:.2f} GB, fp32)")

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        return self.bags[idx], self.labels[idx]


def collate_bags(batch):
    """Variable-size bags -- batch size is always 1 for MIL."""
    features, labels = zip(*batch)
    return features[0], labels[0]


class AttentionMIL(nn.Module):
    """
    Gated Attention-based MIL (Ilse et al. 2018).

    Learns per-patch attention weights via a gated mechanism, aggregates
    the patch embeddings into a fixed-size slide representation, then
    classifies the slide as high or low PRAME.
    """

    def __init__(self, feat_dim, hidden_dim=256, attn_dim=128, dropout=0.25):
        super().__init__()

        self.feature_net = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Gated attention: element-wise product of tanh and sigmoid branches
        self.attention_V = nn.Sequential(
            nn.Linear(hidden_dim, attn_dim),
            nn.Tanh(),
        )
        self.attention_U = nn.Sequential(
            nn.Linear(hidden_dim, attn_dim),
            nn.Sigmoid(),
        )
        self.attention_w = nn.Linear(attn_dim, 1)

        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h = self.feature_net(x)  # (N, hidden_dim)

        a_V = self.attention_V(h)  # (N, attn_dim)
        a_U = self.attention_U(h)  # (N, attn_dim)
        a = self.attention_w(a_V * a_U)  # (N, 1)
        attention = torch.softmax(a, dim=0).squeeze(-1)  # (N,)

        slide_repr = (attention.unsqueeze(-1) * h).sum(dim=0)  # (hidden_dim,)
        logit = self.classifier(slide_repr).squeeze()  # scalar

        return logit, attention

## Training & Evaluation Functions

In [ ]:
def compute_attention_entropy(attention):
    """Shannon entropy of the attention distribution. Higher = more spread."""
    return -(attention * torch.log(attention + 1e-8)).sum()


def train_one_epoch(model, loader, optimizer, criterion, device,
                    entropy_lambda=0.0, grad_clip=0.0, label_smoothing=0.0):
    model.train()
    total_loss = 0
    total_entropy = 0
    preds, truths = [], []

    for features, label in loader:
        true_label = label.item()

        target = label
        if label_smoothing > 0:
            target = label * (1 - 2 * label_smoothing) + label_smoothing

        logit, attention = model(features)
        loss = criterion(logit, target)

        # Attention entropy regularization: maximize entropy (penalize concentration)
        if entropy_lambda > 0:
            entropy = compute_attention_entropy(attention)
            loss = loss - entropy_lambda * entropy
            total_entropy += entropy.item()

        optimizer.zero_grad()
        loss.backward()

        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step()

        total_loss += loss.item()
        preds.append(torch.sigmoid(logit).item())
        truths.append(true_label)

    n = len(loader)
    auc = roc_auc_score(truths, preds)
    mean_entropy = total_entropy / n if entropy_lambda > 0 else 0.0
    return total_loss / n, auc, mean_entropy


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_entropy = 0
    preds, truths = [], []

    with torch.inference_mode():
        for features, label in loader:
            logit, attention = model(features)
            loss = criterion(logit, label)

            total_loss += loss.item()
            total_entropy += compute_attention_entropy(attention).item()
            preds.append(torch.sigmoid(logit).item())
            truths.append(label.item())

    n = len(loader)
    auc = roc_auc_score(truths, preds)
    acc = accuracy_score(truths, [int(p > 0.5) for p in preds])
    mean_entropy = total_entropy / n
    return total_loss / n, auc, acc, preds, truths, mean_entropy


def train_fold(model, train_loader, val_loader, optimizer, scheduler, criterion,
               device, epochs, patience, entropy_lambda=0.0, grad_clip=0.0,
               label_smoothing=0.0, verbose=True, fold_label=""):
    """Train one fold, return metrics history and best model state."""
    best_val_auc = 0
    best_epoch = 0
    patience_counter = 0
    best_state = None

    history = {
        "train_losses": [], "val_losses": [],
        "train_aucs": [], "val_aucs": [],
        "train_entropies": [], "val_entropies": [],
    }

    for epoch in range(epochs):
        train_loss, train_auc, train_ent = train_one_epoch(
            model, train_loader, optimizer, criterion, device,
            entropy_lambda=entropy_lambda, grad_clip=grad_clip,
            label_smoothing=label_smoothing)
        val_loss, val_auc, val_acc, _, _, val_ent = evaluate(
            model, val_loader, criterion, device)
        scheduler.step()

        history["train_losses"].append(train_loss)
        history["val_losses"].append(val_loss)
        history["train_aucs"].append(train_auc)
        history["val_aucs"].append(val_auc)
        history["train_entropies"].append(train_ent)
        history["val_entropies"].append(val_ent)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch + 1
            patience_counter = 0
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            ent_str = f" Ent: {val_ent:.1f}" if entropy_lambda > 0 else ""
            print(f"  {fold_label}Epoch {epoch+1:3d} | "
                  f"Train Loss: {train_loss:.4f} AUC: {train_auc:.3f} | "
                  f"Val Loss: {val_loss:.4f} AUC: {val_auc:.3f} "
                  f"Acc: {val_acc:.3f}{ent_str}")

        if patience_counter >= patience:
            if verbose:
                print(f"  {fold_label}Early stopping at epoch {epoch+1} "
                      f"(best: epoch {best_epoch})")
            break

    history["best_epoch"] = best_epoch
    history["best_val_auc"] = best_val_auc
    return history, best_state

## Load Data & Match Embeddings

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
emb_dir = DRIVE_EMB_DIR / MODEL

slides, labels, patients = [], [], []
missing = 0
for _, row in manifest.iterrows():
    h5_path = emb_dir / row["file_name"].replace(".svs", ".h5")
    if h5_path.exists():
        slides.append(str(h5_path))
        labels.append(row["prame_label"])
        patients.append(row["submitter_id"])
    else:
        missing += 1

slides = np.array(slides)
labels = np.array(labels)
patients = np.array(patients)

# Stratified subsample if requested
if MAX_SLIDES and 0 < MAX_SLIDES < len(slides):
    rng = np.random.RandomState(SEED)
    high_idx = np.where(labels == 1)[0]
    low_idx = np.where(labels == 0)[0]
    n_high = MAX_SLIDES // 2
    n_low = MAX_SLIDES - n_high
    keep = np.concatenate([
        rng.choice(high_idx, min(n_high, len(high_idx)), replace=False),
        rng.choice(low_idx, min(n_low, len(low_idx)), replace=False),
    ])
    slides, labels, patients = slides[keep], labels[keep], patients[keep]
    print(f"Subsampled to {len(slides)} slides (MAX_SLIDES={MAX_SLIDES})")

print(f"Matched: {len(slides)} slides | Missing: {missing}")
print(f"  High PRAME: {labels.sum():.0f} | Low PRAME: {(len(labels) - labels.sum()):.0f}")
print(f"  Unique patients: {len(set(patients))}")

# Sanity check: peek at one embedding
with h5py.File(slides[0], "r") as f:
    print(f"\nSample embedding: {Path(slides[0]).name}")
    print(f"  features shape: {f['features'].shape}, dtype: {f['features'].dtype}")

# Build the GPU-resident cache once. Shared across compare/ablation/full-CV cells.
print("\nBuilding VRAM cache...")
full_dataset = GPUCachedSlideDataset(slides, labels, device)

## Compare Mode (Baseline vs. Regularized — Single Fold)

Run this cell **instead of** the full CV cell to get a quick directional signal.
Trains both configurations on fold 1 with the same data split, then produces
side-by-side comparison plots.

Set the regularization globals above before running.

In [ ]:
# --- Compare Mode: Baseline vs. Regularized on Fold 1 ---

print("=" * 60)
print("COMPARE MODE: Baseline vs. Regularized (Fold 1)")
print("=" * 60)
print(f"  Regularization: entropy_lambda={ENTROPY_LAMBDA}, "
      f"grad_clip={GRAD_CLIP}, label_smoothing={LABEL_SMOOTHING}")

skf = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
train_idx, val_idx = next(iter(skf.split(slides, labels, groups=patients)))

print(f"  Train: {len(train_idx)} slides | Val: {len(val_idx)} slides")

train_patients = set(patients[train_idx])
val_patients = set(patients[val_idx])
assert train_patients.isdisjoint(val_patients), "Patient leakage!"

train_loader = DataLoader(Subset(full_dataset, train_idx.tolist()),
                          batch_size=1, shuffle=True, collate_fn=collate_bags)
val_loader = DataLoader(Subset(full_dataset, val_idx.tolist()),
                        batch_size=1, shuffle=False, collate_fn=collate_bags)

compare_dir = RESULTS_DIR / MODEL / "compare"
compare_dir.mkdir(parents=True, exist_ok=True)

configs = {
    "baseline": {"entropy_lambda": 0.0, "grad_clip": 0.0, "label_smoothing": 0.0},
    "regularized": {"entropy_lambda": ENTROPY_LAMBDA, "grad_clip": GRAD_CLIP,
                     "label_smoothing": LABEL_SMOOTHING},
}

compare_results = {}
for name, cfg in configs.items():
    print(f"\n--- Training: {name} ---")

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = AttentionMIL(FEAT_DIM, HIDDEN_DIM, ATTN_DIM, DROPOUT).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                                 weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history, best_state = train_fold(
        model, train_loader, val_loader, optimizer, scheduler, criterion,
        device, EPOCHS, PATIENCE,
        entropy_lambda=cfg["entropy_lambda"],
        grad_clip=cfg["grad_clip"],
        label_smoothing=cfg["label_smoothing"],
        fold_label=f"[{name}] ")

    # Final eval with best model
    model.load_state_dict(best_state)
    val_loss, val_auc, val_acc, val_preds, val_labels_fold, val_ent = evaluate(
        model, val_loader, criterion, device)

    val_binary = [int(p > 0.5) for p in val_preds]
    tn, fp, fn, tp = confusion_matrix(val_labels_fold, val_binary).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    history["final_val_auc"] = val_auc
    history["final_val_acc"] = val_acc
    history["final_sensitivity"] = sensitivity
    history["final_specificity"] = specificity
    history["final_val_entropy"] = val_ent
    compare_results[name] = history

    print(f"  Best epoch: {history['best_epoch']} | "
          f"Val AUC: {val_auc:.3f} Acc: {val_acc:.3f}")
    print(f"  Sensitivity: {sensitivity:.3f} | Specificity: {specificity:.3f}")
    if val_ent > 0:
        print(f"  Mean attention entropy: {val_ent:.2f}")

# Print comparison table
bl = compare_results["baseline"]
rg = compare_results["regularized"]
print(f"\n{'='*60}")
print("COMPARISON SUMMARY")
print(f"{'='*60}")
print(f"{'Metric':<25} {'Baseline':>10} {'Regularized':>12} {'Delta':>10}")
print(f"{'-'*60}")
for metric, label in [
    ("final_val_auc", "Val AUC"),
    ("final_val_acc", "Val Accuracy"),
    ("final_sensitivity", "Sensitivity"),
    ("final_specificity", "Specificity"),
    ("best_epoch", "Best Epoch"),
    ("final_val_entropy", "Attn Entropy"),
]:
    bv = bl[metric]
    rv = rg[metric]
    delta = rv - bv
    sign = "+" if delta >= 0 else ""
    print(f"  {label:<23} {bv:>10.3f} {rv:>12.3f} {sign}{delta:>9.3f}")

In [ ]:
# --- Compare Mode: Diagnostic Plots ---

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

bl = compare_results["baseline"]
rg = compare_results["regularized"]

# (0,0) Validation AUC curves
ax = axes[0, 0]
ax.plot(bl["val_aucs"], label="Baseline", color="steelblue", lw=2)
ax.plot(rg["val_aucs"], label="Regularized", color="coral", lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation AUC")
ax.set_title("Validation AUC Over Training")
ax.legend()
ax.set_ylim(0, 1)

# (0,1) Training loss curves
ax = axes[0, 1]
ax.plot(bl["train_losses"], label="Baseline train", color="steelblue", lw=1.5)
ax.plot(bl["val_losses"], label="Baseline val", color="steelblue", lw=1.5, linestyle="--")
ax.plot(rg["train_losses"], label="Reg train", color="coral", lw=1.5)
ax.plot(rg["val_losses"], label="Reg val", color="coral", lw=1.5, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Loss Curves (solid=train, dashed=val)")
ax.legend(fontsize=8)

# (1,0) Train-val AUC gap (overfitting indicator)
ax = axes[1, 0]
bl_gap = [t - v for t, v in zip(bl["train_aucs"], bl["val_aucs"])]
rg_gap = [t - v for t, v in zip(rg["train_aucs"], rg["val_aucs"])]
ax.plot(bl_gap, label="Baseline", color="steelblue", lw=2)
ax.plot(rg_gap, label="Regularized", color="coral", lw=2)
ax.axhline(0, color="gray", linestyle="--", lw=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train AUC \u2212 Val AUC")
ax.set_title("Overfitting Gap (lower = less overfitting)")
ax.legend()

# (1,1) Attention entropy over training
ax = axes[1, 1]
if any(e > 0 for e in rg["train_entropies"]):
    ax.plot(bl["val_entropies"], label="Baseline (val)", color="steelblue", lw=2)
    ax.plot(rg["val_entropies"], label="Regularized (val)", color="coral", lw=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Mean Attention Entropy")
    ax.set_title("Attention Spread (higher = more distributed)")
    ax.legend()
else:
    ax.text(0.5, 0.5, "Entropy regularization\nnot enabled",
            ha="center", va="center", fontsize=14, color="gray",
            transform=ax.transAxes)
    ax.set_title("Attention Entropy")

plt.suptitle(f"{MODEL.upper()} \u2014 Baseline vs. Regularized (Fold 1)",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(compare_dir / "compare_baseline_vs_reg.png", dpi=150, bbox_inches="tight")
plt.show()

# Save comparison JSON
comparison = {
    "model": MODEL,
    "mode": "compare",
    "seed": SEED,
    "epochs": EPOCHS,
    "regularization": {
        "entropy_lambda": ENTROPY_LAMBDA,
        "grad_clip": GRAD_CLIP,
        "label_smoothing": LABEL_SMOOTHING,
    },
    "baseline": {k: v for k, v in bl.items()
                 if k.startswith("final_") or k == "best_epoch"},
    "regularized": {k: v for k, v in rg.items()
                    if k.startswith("final_") or k == "best_epoch"},
}
with open(compare_dir / "comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print(f"\nComparison saved to {compare_dir}/")

## Ablation Mode (Each Regularizer in Isolation — Single Fold)

Run this cell group **instead of** the compare cells above when you want to
attribute effects to individual regularizers. Trains 4 configs on the same
fold-1 split:

- `baseline` — all regularizers disabled
- `entropy_only` — only `ENTROPY_LAMBDA` active
- `grad_clip_only` — only `GRAD_CLIP` active
- `label_smooth_only` — only `LABEL_SMOOTHING` active

Costs 4 training runs on fold 1 (~80% the cost of one full 5-fold run).

In [ ]:
# --- Ablation: baseline + each regularizer in isolation on Fold 1 ---

print("=" * 60)
print("ABLATION MODE: Baseline + each regularizer in isolation (Fold 1)")
print("=" * 60)
print(f"  entropy_lambda={ENTROPY_LAMBDA}, grad_clip={GRAD_CLIP}, "
      f"label_smoothing={LABEL_SMOOTHING}")

skf = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
train_idx, val_idx = next(iter(skf.split(slides, labels, groups=patients)))

print(f"  Train: {len(train_idx)} slides | Val: {len(val_idx)} slides")

train_patients = set(patients[train_idx])
val_patients = set(patients[val_idx])
assert train_patients.isdisjoint(val_patients), "Patient leakage!"

train_loader = DataLoader(Subset(full_dataset, train_idx.tolist()),
                          batch_size=1, shuffle=True, collate_fn=collate_bags)
val_loader = DataLoader(Subset(full_dataset, val_idx.tolist()),
                        batch_size=1, shuffle=False, collate_fn=collate_bags)

ablation_dir = RESULTS_DIR / MODEL / "ablation"
ablation_dir.mkdir(parents=True, exist_ok=True)

ablation_configs = {
    "baseline": {"entropy_lambda": 0.0, "grad_clip": 0.0, "label_smoothing": 0.0},
    "entropy_only": {"entropy_lambda": ENTROPY_LAMBDA, "grad_clip": 0.0,
                     "label_smoothing": 0.0},
    "grad_clip_only": {"entropy_lambda": 0.0, "grad_clip": GRAD_CLIP,
                       "label_smoothing": 0.0},
    "label_smooth_only": {"entropy_lambda": 0.0, "grad_clip": 0.0,
                          "label_smoothing": LABEL_SMOOTHING},
}

ablation_histories = {}
for name, cfg in ablation_configs.items():
    print(f"\n--- Training: {name} ---")
    print(f"    (entropy_lambda={cfg['entropy_lambda']}, "
          f"grad_clip={cfg['grad_clip']}, label_smoothing={cfg['label_smoothing']})")

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = AttentionMIL(FEAT_DIM, HIDDEN_DIM, ATTN_DIM, DROPOUT).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                                 weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history, best_state = train_fold(
        model, train_loader, val_loader, optimizer, scheduler, criterion,
        device, EPOCHS, PATIENCE,
        entropy_lambda=cfg["entropy_lambda"],
        grad_clip=cfg["grad_clip"],
        label_smoothing=cfg["label_smoothing"],
        fold_label=f"[{name}] ")

    model.load_state_dict(best_state)
    val_loss, val_auc, val_acc, val_preds, val_labels_fold, val_ent = evaluate(
        model, val_loader, criterion, device)

    val_binary = [int(p > 0.5) for p in val_preds]
    tn, fp, fn, tp = confusion_matrix(val_labels_fold, val_binary).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    history["final_val_auc"] = val_auc
    history["final_val_acc"] = val_acc
    history["final_sensitivity"] = sensitivity
    history["final_specificity"] = specificity
    history["final_val_entropy"] = val_ent
    history["config"] = cfg
    ablation_histories[name] = history

    print(f"  Best epoch: {history['best_epoch']} | "
          f"Val AUC: {val_auc:.3f} Acc: {val_acc:.3f}")
    print(f"  Sensitivity: {sensitivity:.3f} | Specificity: {specificity:.3f}")
    print(f"  Mean attention entropy: {val_ent:.2f}")

# Summary table with deltas from baseline
bl = ablation_histories["baseline"]
print(f"\n{'='*80}")
print("ABLATION SUMMARY (deltas from baseline)")
print(f"{'='*80}")
header = f"{'Metric':<18}" + "".join(f"{name:>16}" for name in ablation_histories)
print(header)
print(f"{'-'*80}")
for metric, label in [
    ("final_val_auc", "Val AUC"),
    ("final_val_acc", "Val Accuracy"),
    ("final_sensitivity", "Sensitivity"),
    ("final_specificity", "Specificity"),
    ("best_epoch", "Best Epoch"),
    ("final_val_entropy", "Attn Entropy"),
]:
    row = f"  {label:<16}"
    for name, h in ablation_histories.items():
        val = h[metric]
        if name == "baseline":
            row += f"{val:>16.3f}"
        else:
            delta = val - bl[metric]
            sign = "+" if delta >= 0 else ""
            row += f"  {val:>6.3f} ({sign}{delta:.3f})"
    print(row)

In [ ]:
# --- Ablation: diagnostic plots (2x2 grid overlaying all 4 configs) ---

colors = {
    "baseline": "steelblue",
    "entropy_only": "coral",
    "grad_clip_only": "seagreen",
    "label_smooth_only": "mediumpurple",
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (0,0) Validation AUC curves
ax = axes[0, 0]
for name, h in ablation_histories.items():
    ax.plot(h["val_aucs"], label=name, color=colors.get(name, "gray"), lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation AUC")
ax.set_title("Validation AUC Over Training")
ax.legend()
ax.set_ylim(0, 1)

# (0,1) Overfitting gap
ax = axes[0, 1]
for name, h in ablation_histories.items():
    gap = [t - v for t, v in zip(h["train_aucs"], h["val_aucs"])]
    ax.plot(gap, label=name, color=colors.get(name, "gray"), lw=2)
ax.axhline(0, color="gray", linestyle="--", lw=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train AUC \u2212 Val AUC")
ax.set_title("Overfitting Gap (lower = less overfitting)")
ax.legend()

# (1,0) Attention entropy over training
ax = axes[1, 0]
for name, h in ablation_histories.items():
    ax.plot(h["val_entropies"], label=name, color=colors.get(name, "gray"), lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean Attention Entropy (val)")
ax.set_title("Attention Spread (higher = more distributed)")
ax.legend()

# (1,1) Final metrics bar chart
ax = axes[1, 1]
metric_names = ["Val AUC", "Val Acc", "Sensitivity", "Specificity"]
metric_keys = ["final_val_auc", "final_val_acc",
               "final_sensitivity", "final_specificity"]
config_names = list(ablation_histories.keys())
x = np.arange(len(metric_names))
width = 0.8 / len(config_names)
for i, name in enumerate(config_names):
    values = [ablation_histories[name][k] for k in metric_keys]
    offset = (i - (len(config_names) - 1) / 2) * width
    ax.bar(x + offset, values, width, label=name,
           color=colors.get(name, "gray"), alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylabel("Score")
ax.set_title("Final Metrics (best epoch)")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)

plt.suptitle(f"{MODEL.upper()} \u2014 Regularization Ablation (Fold 1)",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(ablation_dir / "ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Save ablation JSON
ablation_out = {
    "model": MODEL,
    "mode": "ablation",
    "seed": SEED,
    "epochs": EPOCHS,
    "configs": {
        name: {
            **h["config"],
            "best_epoch": h["best_epoch"],
            "final_val_auc": h["final_val_auc"],
            "final_val_acc": h["final_val_acc"],
            "final_sensitivity": h["final_sensitivity"],
            "final_specificity": h["final_specificity"],
            "final_val_entropy": h["final_val_entropy"],
        }
        for name, h in ablation_histories.items()
    },
}
with open(ablation_dir / "ablation.json", "w") as f:
    json.dump(ablation_out, f, indent=2)

print(f"\nAblation saved to {ablation_dir}/")

### Post-Ablation Heuristic

Based on the fold-1 ablation (100 slides, 50 epochs):

| Regularizer | Verdict | Recommended Value |
|---|---|---|
| `ENTROPY_LAMBDA` | **Drop.** `entropy_only` and `baseline` runs are bit-identical — the unregularized attention already sits near max entropy, so the reg term has nothing to do. | `0.0` |
| `GRAD_CLIP` | **Keep.** Measurably reduces the train–val AUC overfitting gap. Zero cost-function impact. Cheap stability insurance for variable bag sizes. | `1.0` |
| `LABEL_SMOOTHING` | **Keep.** Only regularizer with a positive AUC delta (+0.02). Calibration shift is a feature for Component 2's confidence-based routing. | `0.05` |

**For full 5-fold CV:** set `GRAD_CLIP = 1.0` and `LABEL_SMOOTHING = 0.05` in the Configuration cell, leave `ENTROPY_LAMBDA = 0.0`, then run the Full Cross-Validation cells below.

**Caveat:** 40-slide val set means ±1 sample flip = ±0.05 on sens/spec. Deltas are within noise floor — treat as directional.

## Full Cross-Validation Training Loop

Run this cell **instead of** the compare cells above for the full 5-fold evaluation.
Uses whatever regularization is set in the Configuration cell.

In [ ]:
skf = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

out_dir = RESULTS_DIR / MODEL
out_dir.mkdir(parents=True, exist_ok=True)

fold_results = []
all_val_preds = np.zeros(len(slides))
all_val_labels = np.zeros(len(slides))
val_indices_all = []

all_train_losses, all_val_losses = [], []
all_train_aucs, all_val_aucs = [], []

for fold, (train_idx, val_idx) in enumerate(
        skf.split(slides, labels, groups=patients)):

    print(f"\n{'='*50}")
    print(f"Fold {fold + 1}/{FOLDS}")
    print(f"  Train: {len(train_idx)} slides | Val: {len(val_idx)} slides")

    # Verify no patient leakage
    train_patients = set(patients[train_idx])
    val_patients = set(patients[val_idx])
    assert train_patients.isdisjoint(val_patients), "Patient leakage!"

    train_loader = DataLoader(Subset(full_dataset, train_idx.tolist()),
                              batch_size=1, shuffle=True,
                              collate_fn=collate_bags)
    val_loader = DataLoader(Subset(full_dataset, val_idx.tolist()),
                            batch_size=1, shuffle=False,
                            collate_fn=collate_bags)

    model = AttentionMIL(FEAT_DIM, HIDDEN_DIM, ATTN_DIM, DROPOUT).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                                 weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history, best_state = train_fold(
        model, train_loader, val_loader, optimizer, scheduler, criterion,
        device, EPOCHS, PATIENCE,
        entropy_lambda=ENTROPY_LAMBDA,
        grad_clip=GRAD_CLIP,
        label_smoothing=LABEL_SMOOTHING)

    all_train_losses.append(history["train_losses"])
    all_val_losses.append(history["val_losses"])
    all_train_aucs.append(history["train_aucs"])
    all_val_aucs.append(history["val_aucs"])

    # Evaluate best model on val set
    model.load_state_dict(best_state)
    _, val_auc, val_acc, val_preds, val_labels_fold, _ = evaluate(
        model, val_loader, criterion, device)

    val_binary = [int(p > 0.5) for p in val_preds]
    tn, fp, fn, tp = confusion_matrix(val_labels_fold, val_binary).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    print(f"  Best epoch: {history['best_epoch']} | "
          f"Val AUC: {val_auc:.3f} Acc: {val_acc:.3f}")
    print(f"  Sensitivity: {sensitivity:.3f} | "
          f"Specificity: {specificity:.3f}")

    fold_results.append({
        "fold": fold + 1,
        "best_epoch": history["best_epoch"],
        "val_auc": val_auc,
        "val_acc": val_acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "train_size": len(train_idx),
        "val_size": len(val_idx),
    })

    for i, idx in enumerate(val_idx):
        all_val_preds[idx] = val_preds[i]
        all_val_labels[idx] = val_labels_fold[i]
    val_indices_all.extend(val_idx.tolist())

    # Save best model weights to Drive
    torch.save(best_state, out_dir / f"fold{fold+1}_model.pt")

print(f"\nTraining complete. Models saved to {out_dir}/")

## Aggregate Results & Plots

In [ ]:
aucs = [r["val_auc"] for r in fold_results]
accs = [r["val_acc"] for r in fold_results]
sens = [r["sensitivity"] for r in fold_results]
specs = [r["specificity"] for r in fold_results]

val_idx_arr = np.array(val_indices_all)
pooled_auc = roc_auc_score(all_val_labels[val_idx_arr],
                           all_val_preds[val_idx_arr])

print(f"{'='*50}")
print(f"AGGREGATE RESULTS -- {MODEL.upper()}")
print(f"{'='*50}")
print(f"AUC:         {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")
print(f"Accuracy:    {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
print(f"Sensitivity: {np.mean(sens):.3f} +/- {np.std(sens):.3f}")
print(f"Specificity: {np.mean(specs):.3f} +/- {np.std(specs):.3f}")
print(f"Pooled AUC:  {pooled_auc:.3f}")

In [ ]:
# --- Per-fold AUC bar chart + Pooled ROC curve ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.bar(range(1, FOLDS + 1), aucs, color="steelblue", alpha=0.8)
ax.axhline(np.mean(aucs), color="red", linestyle="--",
           label=f"Mean: {np.mean(aucs):.3f}")
ax.set_xlabel("Fold")
ax.set_ylabel("AUC")
ax.set_title(f"{MODEL.upper()} -- Validation AUC per Fold")
ax.set_ylim(0, 1)
ax.legend()

ax = axes[1]
pool_labels = all_val_labels[val_idx_arr]
pool_preds = all_val_preds[val_idx_arr]
fpr, tpr, _ = roc_curve(pool_labels, pool_preds)
ax.plot(fpr, tpr, color="steelblue", lw=2,
        label=f"Pooled AUC = {pooled_auc:.3f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL.upper()} -- Pooled ROC Curve")
ax.legend()

plt.tight_layout()
plt.savefig(out_dir / "cv_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Training curves (loss + AUC per fold) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for fold_i, (tl, vl) in enumerate(zip(all_train_losses, all_val_losses)):
    axes[0].plot(tl, alpha=0.5, label=f"Fold {fold_i+1} train")
    axes[0].plot(vl, alpha=0.5, linestyle="--", label=f"Fold {fold_i+1} val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title(f"{MODEL.upper()} -- Loss Curves")
axes[0].legend(fontsize=7)

for fold_i, (ta, va) in enumerate(zip(all_train_aucs, all_val_aucs)):
    axes[1].plot(ta, alpha=0.5, label=f"Fold {fold_i+1} train")
    axes[1].plot(va, alpha=0.5, linestyle="--", label=f"Fold {fold_i+1} val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUC")
axes[1].set_title(f"{MODEL.upper()} -- AUC Curves")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(out_dir / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Save Results to Drive

In [ ]:
# Save per-fold CSV and summary JSON
results_df = pd.DataFrame(fold_results)
results_df.to_csv(out_dir / "cv_results.csv", index=False)

summary = {
    "model": MODEL,
    "folds": FOLDS,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "hidden_dim": HIDDEN_DIM,
    "attn_dim": ATTN_DIM,
    "dropout": DROPOUT,
    "patience": PATIENCE,
    "seed": SEED,
    "entropy_lambda": ENTROPY_LAMBDA,
    "grad_clip": GRAD_CLIP,
    "label_smoothing": LABEL_SMOOTHING,
    "num_slides": len(slides),
    "num_patients": len(set(patients)),
    "mean_auc": float(np.mean(aucs)),
    "std_auc": float(np.std(aucs)),
    "mean_acc": float(np.mean(accs)),
    "std_acc": float(np.std(accs)),
    "mean_sensitivity": float(np.mean(sens)),
    "std_sensitivity": float(np.std(sens)),
    "mean_specificity": float(np.mean(specs)),
    "std_specificity": float(np.std(specs)),
    "pooled_auc": float(pooled_auc),
}
with open(out_dir / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to Drive:")
print(f"  {out_dir}/cv_results.csv")
print(f"  {out_dir}/summary.json")
print(f"  {out_dir}/cv_results.png")
print(f"  {out_dir}/training_curves.png")
print(f"  {out_dir}/fold*_model.pt")
print()
print(results_df.to_string(index=False))